In [1]:
from pathlib import Path
import shutil

BASE_DIR = Path("/home/hemangpc/code/ocr_to_text_pdf")

INPUT_DIR = BASE_DIR / "input"
UPLOAD_DIR = BASE_DIR / "uploads"

INPUT_DIR.mkdir(exist_ok=True)
UPLOAD_DIR.mkdir(exist_ok=True)

pdf_files = list(INPUT_DIR.glob("*.pdf"))

if not pdf_files:
    print("No PDF file found in the input folder.")
else:
    for pdf_path in pdf_files:
        stored_path = UPLOAD_DIR / pdf_path.name
        shutil.copy2(pdf_path, stored_path)

        print(f"Stored: {pdf_path.name} -> {stored_path}")

Stored: SI_CHRONICLES_25.pdf -> /home/hemangpc/code/ocr_to_text_pdf/uploads/SI_CHRONICLES_25.pdf


In [ ]:
from pdf2image import convert_from_path

if not pdf_files:
    print("No PDF file found in uploads folder.")
else:
    for pdf_path in pdf_files:
        output_image_dir = BASE_DIR / f"{pdf_path.stem}_images"
        output_image_dir.mkdir(exist_ok=True)

        pages = convert_from_path(pdf_path, dpi=300)

        for page_number, page_image in enumerate(pages, start=1):
            image_path = output_image_dir / f"page_{page_number}.png"
            page_image.save(image_path, "PNG")

        print(f"Converted {pdf_path.name} into images at: {output_image_dir}")

In [ ]:
import gc
from pdf2image import pdfinfo_from_path

image_folder = output_image_dir
cleaned_folder = BASE_DIR / f"{image_folder.name}_cleaned"
cleaned_folder.mkdir(exist_ok=True)

DPI = 300  # match cell 2's render DPI if you're re-rendering rather than reading cell 2's saved PNGs

num_pages = pdfinfo_from_path(pdf_path)["Pages"]

for page_number in range(1, num_pages + 1):
    # render exactly one page - never holds the whole PDF in memory at once
    page_image = convert_from_path(
        pdf_path, dpi=DPI, first_page=page_number, last_page=page_number
    )[0]

    page_image = ImageOps.grayscale(page_image)
    page_image = ImageOps.autocontrast(page_image)
    page_image = page_image.filter(ImageFilter.MedianFilter(size=3))

    image_array = np.array(page_image)
    cleaned_array = cv2.adaptiveThreshold(
        image_array, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )

    cleaned_path = cleaned_folder / f"page_{page_number}.png"
    Image.fromarray(cleaned_array).save(cleaned_path)

    # release this page before the next iteration starts
    page_image.close()
    del image_array, cleaned_array
    gc.collect()

    print(f"Cleaned page {page_number}/{num_pages} -> {cleaned_path}")